## Dataset Preparation - Part 1

<span style="color:green">

**This notebook corresponds to Step 1 of the data preparation covered in Practical 8 (i.e., `8_1_data_prep_part1.ipynb`). Therefore, in Practical 9, this step can be skipped, and all data can be copied directly from the "processed" folde in practical 8 to the "processed" folder in practicals 9 for downstream tasks.**

</span>

In this notebook, we prepare the dataset by converting DICOM images to NIfTI files along with their corresponding label masks.

We use a small dataset of 3D CT scans from **20 patients** (10 women and 10 men), with **75% of the cases** featuring tumors. The dataset includes multiple segmentation masks such as liver, bone, and spleen masks (not all masks are available for every case). 

Since the dataset files for each case are provided in a zip format and the DICOM files lack extensions, we use a script called **`extract_and_rename_3d_ircadb.py`** to extract and rename the files. In this notebook, we then:
- Select cases containing a specified label (e.g., "liver").
- Convert the selected cases to NIfTI files for easier use and visualization in the next sections.

The dataset can be downloaded in NIfTI format from:  
[IRCAD Liver Segmentation Dataset](https://www.ircad.fr/research/data-sets/liver-segmentation-3d-ircadb-01/)  

For convenience, both the raw and processed datasets are also available in the `data` directory.

**Dataset License**: ([Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International License](https://creativecommons.org/licenses/by-nc-nd/4.0/)).

---

### 1. Import Libraries
Let's start by importing the necessary libraries.


In [1]:
import os
import shutil
import numpy as np
import nibabel as nib
from tqdm import tqdm
import dicom2nifti

---
### 2. Define Paths
Here, we specify:
- **`src_dir`**: Path to the source directory containing the DICOM files.
- **`out_dir`**: Path to the output directory where the NIfTI files will be saved.

We are particularly interested in cases containing the "liver" mask.

In [2]:
root_dir = "data/processed"

src_dir = os.path.join(root_dir, "3d_ircadb_images_masks_all_dicom")
out_dir = os.path.join(root_dir, "3d_ircadb__nifti_liver")

mask_name = "liver"

#### 2.1 Reset output directory

In [3]:
# Reset the output directory to ensure no leftover files
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)

os.makedirs(out_dir, exist_ok=True)

---
### 3. List File Names

Here, we list all subject directories in the source folder. Sorting helps ensure consistent processing order.

In [4]:
subjects = sorted(os.listdir(src_dir))

print('Number of subjects:', len(subjects))

Number of subjects: 20


---
### 4. Rename Subjects

To standardize file names, we will:
1. Convert names to lowercase.
2. Zero-pad numeric parts to ensure consistent sorting.

In [5]:
def format_name(file_name):
    """Format the filename by converting to lowercase and zero-padding numeric part."""
    parts = file_name.split(".")
    return f"{parts[0].lower()}_{int(parts[1]):02}"

sample_name = subjects[0]
new_name = format_name(sample_name)

print(f"{sample_name} ===> {new_name}")

3Dircadb1.1 ===> 3dircadb1_01


---
### 5. Convert DICOM to NIfTI

For each subject:
1. Identify the DICOM folder containing the images and masks.
2. Use `dicom2nifti.dicom_series_to_nifti` to convert DICOM files to NIfTI format.

- **`img_path`**: Path to the DICOM folder with CT images.
- **`mask_path`**: Path to the DICOM folder with masks.

The converted NIfTI files will be saved in the output directory with consistent naming conventions. <br>
Masks will be post-processed to ensure they are binary (0 and 1) and stored as uint8 data type.

In [6]:
for subject in tqdm(subjects, desc="Converting DICOM to NIfTI"):
    img_path = os.path.join(src_dir, subject, "PATIENT_DICOM")
    mask_path = os.path.join(src_dir, subject, "MASKS_DICOM", mask_name)

    if os.path.exists(mask_path):
        formatted_name = format_name(subject)

        # Convert the image
        nifti_img_path = os.path.join(out_dir, f"{formatted_name}_image.nii.gz")
        dicom2nifti.dicom_series_to_nifti(img_path, nifti_img_path, reorient_nifti=False)

        # Convert the mask
        nifti_mask_path = os.path.join(out_dir, f"{formatted_name}_mask_{mask_name}.nii.gz")
        dicom2nifti.dicom_series_to_nifti(mask_path, nifti_mask_path, reorient_nifti=False)

        # Load the converted mask NIfTI file
        mask_nii = nib.load(nifti_mask_path)
        mask_data = mask_nii.get_fdata()

        # Ensure binary values and correct data type
        mask_data = (mask_data > 0).astype(np.uint8)

        # Save the corrected mask back to a NIfTI file
        corrected_mask_nii = nib.Nifti1Image(mask_data, mask_nii.affine, mask_nii.header)
        corrected_mask_nii.header.set_data_dtype(np.uint8)  # Ensure header is updated to uint8
        nib.save(corrected_mask_nii, nifti_mask_path)

Converting DICOM to NIfTI: 100%|██████████| 20/20 [00:58<00:00,  2.91s/it]
